# Convergence, Termination, and Reporting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ahmed-Bayoumy/MECH559/blob/DEV/L09/L09_convergence_and_termination.ipynb)
### MECH 559 - Systems Optimization - L09

Companion notebook to the closing section of the "Algorithms for nonlinearly constrained optimization" lecture: how do we know an algorithm is *working* (convergence rate), when do we let it *stop* (termination criteria), and what should we *report*? We use Himmelblau's function - a classic four-minima test function - for the convergence-rate and multi-start demonstrations, then return to the barrier/penalty example for the termination-criteria and reporting demonstrations.

## Learning objectives

By the end, students should be able to:

1. Distinguish linear, superlinear, and quadratic local convergence numerically, not just by formula.
2. Explain why gradient-based methods need multiple initial guesses (multi-start) when the problem is multimodal.
3. Compute the *correct* constrained termination metric $\|\nabla f+\nabla h^\mathsf{T}\lambda+\nabla g^\mathsf{T}\mu\|$ and see how the choice of $\epsilon$ changes when an algorithm stops.
4. Assemble a short, complete "statistics of the run" report as recommended in the slides.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

plt.rcParams.update({
    "font.size": 11,
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

COLORS = {"linear": "#D1495B", "superlinear": "#EDAE49", "quadratic": "#1D5FA8", "optimum": "#2A9D5C"}
print(f"NumPy {np.__version__}, SciPy ok")

NumPy 2.5.0, SciPy ok


---
## 1. Convergence rate, made concrete

Himmelblau's function $f(x_1,x_2)=(x_1^2+x_2-11)^2+(x_1+x_2^2-7)^2$ has four global minima with $f=0$; we start close to $(3,2)$ so all three methods below converge to the *same* point, letting us compare rates directly.

In [2]:
def f(x):
    return (x[0] ** 2 + x[1] - 11) ** 2 + (x[0] + x[1] ** 2 - 7) ** 2

def grad_f(x):
    x1, x2 = x
    return np.array([
        4 * x1 * (x1 ** 2 + x2 - 11) + 2 * (x1 + x2 ** 2 - 7),
        2 * (x1 ** 2 + x2 - 11) + 4 * x2 * (x1 + x2 ** 2 - 7),
    ])

def hess_f(x):
    x1, x2 = x
    d2x1 = 12 * x1 ** 2 + 4 * x2 - 42
    d2x2 = 4 * x1 + 12 * x2 ** 2 - 26
    dxy = 4 * x1 + 4 * x2
    return np.array([[d2x1, dxy], [dxy, d2x2]])

MINIMA = np.array([(3.0, 2.0), (-2.805118, 3.131312), (-3.779310, -3.283186), (3.584428, -1.848126)])
x_star = MINIMA[0]
x0 = [4.0, 3.0]

def gradient_descent(x0, alpha=0.01, n=60):
    x = np.array(x0, dtype=float)
    traj = [x.copy()]
    for _ in range(n):
        x = x - alpha * grad_f(x)
        traj.append(x.copy())
    return np.array(traj)

def newton(x0, n=15):
    x = np.array(x0, dtype=float)
    traj = [x.copy()]
    for _ in range(n):
        x = x + np.linalg.solve(hess_f(x), -grad_f(x))
        traj.append(x.copy())
    return np.array(traj)

traj_bfgs = [np.array(x0, dtype=float)]
minimize(f, x0, jac=grad_f, method="BFGS", callback=lambda xk: traj_bfgs.append(xk.copy()),
         options={"gtol": 1e-12})
traj_bfgs = np.array(traj_bfgs)

traj_gd = gradient_descent(x0)
traj_nt = newton(x0)

print(f"iterations run (GD, Newton: fixed budget; BFGS: its own gtol) -- "
      f"GD: {len(traj_gd)-1}, BFGS: {len(traj_bfgs)-1}, Newton: {len(traj_nt)-1}")

iterations run (GD, Newton: fixed budget; BFGS: its own gtol) -- GD: 60, BFGS: 13, Newton: 15


Compare the error norms on a log scale, and empirically check the defining ratios from the slides:
linear ($e_{k+1}/e_k\to c<1$), quadratic ($e_{k+1}/e_k^2\to c$), superlinear ($e_{k+1}/e_k\to0$).

In [3]:
def err(traj):
    return np.linalg.norm(traj - x_star, axis=1)

e_gd, e_nt, e_bfgs = err(traj_gd), err(traj_nt), err(traj_bfgs)

fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.semilogy(e_gd, "-o", ms=4, color=COLORS["linear"], label="gradient descent (linear)")
ax.semilogy(e_bfgs, "-s", ms=5, color=COLORS["superlinear"], label="BFGS (superlinear)")
ax.semilogy(e_nt[e_nt > 1e-16], "-^", ms=6, color=COLORS["quadratic"], label="Newton (quadratic)")
ax.set(xlabel="iteration $k$", ylabel=r"$\|x_k-x^\star\|$", title="Three convergence rates, same starting point and target")
ax.legend()
plt.tight_layout()
plt.show()

print("linear check       e_(k+1)/e_k   (GD)      :", np.round(e_gd[1:11] / e_gd[:10], 3))
print("superlinear check  e_(k+1)/e_k   (BFGS)     :", np.round(e_bfgs[1:9] / e_bfgs[:8], 3), " -> ratio itself shrinks to 0")
mask = e_nt[:-1] > 1e-9
print("quadratic check    e_(k+1)/e_k^2 (Newton)   :", np.round(e_nt[1:][mask] / e_nt[:-1][mask] ** 2, 3))
n_digits = -np.log10(e_nt[e_nt > 0])
print("correct digits per Newton iterate           :", np.round(n_digits, 2), "(roughly doubling each step)")

linear check       e_(k+1)/e_k   (GD)      : [0.295 0.57  0.671 0.714 0.725 0.73  0.733 0.736 0.738 0.739]
superlinear check  e_(k+1)/e_k   (BFGS)     : [0.343 0.702 0.694 0.109 0.079 0.025 0.334 0.135]  -> ratio itself shrinks to 0
quadratic check    e_(k+1)/e_k^2 (Newton)   : [0.202 0.337 0.678 0.812 0.757]
correct digits per Newton iterate           : [-0.15  0.39  1.26  2.69  5.46 11.04] (roughly doubling each step)


C:\Users\ahbay\AppData\Local\Temp\ipykernel_72484\3775453811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Away from a nearly-quadratic function like the circle example in `L09_sqp.ipynb`, the ratio $e_{k+1}/e_k^2$ need not settle to one fixed constant this early - what identifies quadratic convergence is that the ratio stays **bounded** (does not blow up) while the number of correct digits **roughly doubles every iteration**, exactly as printed above.

---
## 2. Multi-start: why the initial guess matters

Himmelblau's function has **four** local (and global) minima. A gradient-based method run once will converge to whichever basin it starts in - a design-of-experiments sample of initial guesses (multi-start) is needed to have confidence that the *best* minimum has been found.

In [4]:
rng = np.random.default_rng(559)
starts = rng.uniform(-5.0, 5.0, size=(60, 2))

results = []
for x0s in starts:
    res = minimize(f, x0s, jac=grad_f, method="BFGS")
    dists = np.linalg.norm(MINIMA - res.x, axis=1)
    basin = np.argmin(dists) if dists.min() < 1e-3 else -1
    results.append((x0s, res.x, basin))

basins = np.array([r[2] for r in results])
for i, m in enumerate(MINIMA):
    print(f"minimum {i} at {np.round(m,3)}: reached from {np.sum(basins==i)}/{len(starts)} starts")
print(f"failed to converge to any known minimum: {np.sum(basins==-1)}/{len(starts)}")

minimum 0 at [3. 2.]: reached from 16/60 starts
minimum 1 at [-2.805  3.131]: reached from 18/60 starts
minimum 2 at [-3.779 -3.283]: reached from 13/60 starts
minimum 3 at [ 3.584 -1.848]: reached from 13/60 starts
failed to converge to any known minimum: 0/60


In [5]:
fig, ax = plt.subplots(figsize=(7, 6))
palette = ["#1D5FA8", "#D1495B", "#2A9D5C", "#EDAE49"]
for i in range(4):
    mask = basins == i
    ax.scatter(starts[mask, 0], starts[mask, 1], color=palette[i], s=35, alpha=0.8, label=f"-> minimum {i}")
for i, m in enumerate(MINIMA):
    ax.plot(*m, "*", color=palette[i], ms=22, mec="black", mew=1)
ax.set(xlabel="$x_1^{(0)}$", ylabel="$x_2^{(0)}$", title="Which minimum each starting point converges to")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

C:\Users\ahbay\AppData\Local\Temp\ipykernel_72484\2862411426.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 3. Termination criteria on a constrained problem

Back to the constrained example from `L09_augmented_lagrangian_method.ipynb`: $f=3x_1^2+x_2^2$, $g_1=2-x_2-x_1\le0$, $x^\star=(0.5,1.5)$, $\mu^\star=3$. At every outer augmented-Lagrangian iteration we log all three candidate stopping metrics from the slides:

$$
\|\nabla f+\mu\nabla g_1\|\le\epsilon,\qquad |f_{k+1}-f_k|\le\epsilon,\qquad \|x_{k+1}-x_k\|\le\epsilon.
$$

(Note this is the corrected form $\|\nabla f+\nabla g^\mathsf{T}\mu\|$ - the *gradient* of the Lagrangian, not the constraint value itself.)

In [6]:
def fc(x):
    return 3.0 * x[0] ** 2 + x[1] ** 2

def grad_fc(x):
    return np.array([6.0 * x[0], 2.0 * x[1]])

def g1(x):
    return 2.0 - x[1] - x[0]

grad_g1 = np.array([-1.0, -1.0])

def T_aug(x, r, mu):
    return fc(x) + mu * g1(x) + (1.0 / r) * max(0.0, g1(x)) ** 2

x = np.array([0.0, 0.0])
mu, r = 0.0, 1.0
log = []
for k in range(9):
    res = minimize(T_aug, x, args=(r, mu), method="BFGS")
    x_new = res.x
    lagrangian_grad_norm = np.linalg.norm(grad_fc(x_new) + mu * grad_g1)
    log.append({
        "k": k, "x": x_new.copy(), "f": fc(x_new),
        "d_f": abs(fc(x_new) - fc(x)) if k > 0 else np.nan,
        "d_x": np.linalg.norm(x_new - x),
        "lagrangian_grad": lagrangian_grad_norm,
    })
    mu = mu + (2.0 / r) * g1(x_new)
    x = x_new

print(f"{'k':>2} {'|f_k+1 - f_k|':>15} {'||x_k+1 - x_k||':>16} {'||grad L||':>12}")
for entry in log:
    print(f"{entry['k']:>2} {entry['d_f']:>15.3e} {entry['d_x']:>16.3e} {entry['lagrangian_grad']:>12.3e}")

 k   |f_k+1 - f_k|  ||x_k+1 - x_k||   ||grad L||
 0             nan        9.035e-01    2.424e+00
 1       1.020e+00        3.872e-01    1.039e+00
 2       5.471e-01        1.660e-01    4.453e-01
 3       2.547e-01        7.112e-02    1.908e-01
 4       1.129e-01        3.048e-02    8.179e-02
 5       4.906e-02        1.306e-02    3.505e-02
 6       2.115e-02        5.599e-03    1.502e-02
 7       9.088e-03        2.399e-03    6.438e-03
 8       3.899e-03        1.028e-03    2.759e-03


Now ask: **at which iteration would the algorithm have stopped** for a few choices of $\epsilon$, under each of the three criteria?

In [7]:
def first_below(values, eps):
    for entry in values:
        v = entry
        if not np.isnan(v) and v <= eps:
            return True
    return False

d_f = [e["d_f"] for e in log]
d_x = [e["d_x"] for e in log]
grad_L = [e["lagrangian_grad"] for e in log]

print(f"{'epsilon':>10} {'stop on |df|<=eps at k=':>26} {'stop on ||dx||<=eps at k=':>28} {'stop on ||gradL||<=eps at k=':>30}")
for eps in [1e-1, 1e-2, 1e-3, 1e-4]:
    k_f = next((e["k"] for e in log if not np.isnan(e["d_f"]) and e["d_f"] <= eps), None)
    k_x = next((e["k"] for e in log if e["d_x"] <= eps), None)
    k_g = next((e["k"] for e in log if e["lagrangian_grad"] <= eps), None)
    print(f"{eps:>10.0e} {str(k_f):>26} {str(k_x):>28} {str(k_g):>30}")

   epsilon    stop on |df|<=eps at k=    stop on ||dx||<=eps at k=   stop on ||gradL||<=eps at k=
     1e-01                          5                            3                              4
     1e-02                          7                            6                              7
     1e-03                       None                         None                           None
     1e-04                       None                         None                           None


The three criteria do **not** agree on when to stop, and a looser $\epsilon$ stops earlier at a less accurate (and, for $\|\nabla L\|$, less *feasible*) point - exactly the slide's warning that "the choice of value of $\epsilon$ will affect results.\"

---
## 4. "Statistics of an optimization run" - a minimal report

The slides recommend reporting the initial guess, tolerances, iteration/eval counts, active constraints and multipliers, and (if applicable) parametric studies. Here is that report for the final augmented-Lagrangian run above.

In [8]:
def optimization_report(x0, eps, log, active_tol=1e-6):
    xf = log[-1]["x"]
    active = g1(xf) > -active_tol
    report = f'''Optimization run report
------------------------
Initial guess       : x0 = {x0}
Tolerance            : eps = {eps:.1e}  (on ||grad L||)
Outer iterations     : {len(log)}
Final point          : x* = {np.round(xf, 6)}
Final objective      : f* = {log[-1]['f']:.6f}
Active constraints   : g1 {'ACTIVE' if active else 'inactive'} (g1 = {g1(xf):.2e})
Multiplier estimate  : mu ~ {mu:.4f}  (exact mu* = 3.0)
Termination metric   : ||grad L|| = {log[-1]['lagrangian_grad']:.2e}
'''
    return report

print(optimization_report(x0=[0.0, 0.0], eps=1e-3, log=log))

Optimization run report
------------------------
Initial guess       : x0 = [0.0, 0.0]
Tolerance            : eps = 1.0e-03  (on ||grad L||)
Outer iterations     : 9
Final point          : x* = [0.499756 1.499268]
Final objective      : f* = 2.997074
Active constraints   : g1 ACTIVE (g1 = 9.76e-04)
Multiplier estimate  : mu ~ 2.9985  (exact mu* = 3.0)
Termination metric   : ||grad L|| = 2.76e-03



---
## Takeaways

1. "Linear", "superlinear", and "quadratic" are not just labels - they are visibly, measurably different behaviors of the same error norm across iterations, and the empirical ratios ($e_{k+1}/e_k$ or $e_{k+1}/e_k^2$) confirm which regime a given method is in.
2. A gradient-based method converges to *a* local minimizer, not *the* global one; multi-start with a DoE-generated set of initial guesses is how confidence in global optimality is built for multimodal problems.
3. The mathematically correct constrained termination test is the norm of the *gradient of the Lagrangian*, $\|\nabla f+\nabla h^\mathsf{T}\lambda+\nabla g^\mathsf{T}\mu\|$ - not the constraint values themselves - and different termination criteria (and different $\epsilon$) genuinely disagree about when to stop.
4. A complete report of an optimization run needs more than just the final $x^\star$: initial guess, tolerances, iteration counts, active constraints, and multiplier values are all part of communicating whether a result should be trusted.